# 05 Evaluation\n
Evaluate quality with automatic and finance-specific checks.

In [ ]:
import json
import torch
import numpy as np
from pathlib import Path
from datasets import Dataset

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

from rouge_score import rouge_scorer
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports successful")

# Load test set
test_path = Path('../data/processed/test.jsonl')
print(f"\nTest set exists: {test_path.exists()}")

test_data = []
if test_path.exists():
    with open(test_path, 'r') as f:
        test_data = [json.loads(line) for line in f]
    print(f"Loaded {len(test_data)} test samples")

In [ ]:
print("\n" + "="*60)
print("Evaluation Metrics")
print("="*60)

# ROUGE scores
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

rouge1_scores = []
rougeL_scores = []

for pred, ref in zip(predictions, references):
    scores = scorer.score(ref, pred)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

print(f"\nROUGE Scores (higher is better):")
print(f"  ROUGE-1: {np.mean(rouge1_scores):.4f} ± {np.std(rouge1_scores):.4f}")
print(f"  ROUGE-L: {np.mean(rougeL_scores):.4f} ± {np.std(rougeL_scores):.4f}")

# Length comparison
pred_lengths = [len(p.split()) for p in predictions]
ref_lengths = [len(r.split()) for r in references]

print(f"\nResponse Length Comparison:")
print(f"  Predicted (avg): {np.mean(pred_lengths):.1f} words")
print(f"  Reference (avg): {np.mean(ref_lengths):.1f} words")

# Quality assessment (subjective)
print(f"\nQualitative Assessment:")
print(f"  Generated {len(predictions)} answers")
print(f"  Average context length: {np.mean([len(s['context'].split()) for s in eval_samples]):.1f} words")

## 4. Compute Metrics

In [ ]:
print("\nGenerating predictions on test set...")

# Take first 10 samples for evaluation (to avoid long runtime)
eval_samples = test_data[:10]
predictions = []
references = []

for idx, sample in enumerate(eval_samples):
    instruction = sample['instruction']
    context = sample['context']
    reference = sample['output']
    
    # Create prompt
    prompt = f"""[INST] Answer the financial question based on the provided context.

Context: {context}

Question: {instruction} [/INST]

Answer:"""
    
    # Generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            top_p=0.9,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the answer part (after "Answer:")
    answer_start = prediction.find("Answer:") + len("Answer:")
    answer = prediction[answer_start:].strip() if answer_start > len("Answer:") else prediction
    
    predictions.append(answer)
    references.append(reference)
    
    print(f"\n{idx+1}. Q: {instruction[:100]}...")
    print(f"   Predicted: {answer[:100]}...")
    print(f"   Reference: {reference[:100]}...")

print(f"\n✓ Generated {len(predictions)} predictions")

## 3. Generate Predictions

In [ ]:
print("="*60)
print("Loading Fine-Tuned Model")
print("="*60)

BASE_MODEL = "mistralai/Mistral-7B"
ADAPTER_PATH = "../models/finance-adapter-pilot"

# Check if adapter exists
adapter_path = Path(ADAPTER_PATH)
if not adapter_path.exists():
    print(f"⚠ Adapter not found at {ADAPTER_PATH}")
    print("Run 03_qlora_training_pilot.ipynb first")
else:
    print(f"\nLoading base model: {BASE_MODEL}")
    
    # BitsAndBytes config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    # Load base model
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    # Load adapter
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    model.eval()
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
    
    print(f"✓ Model loaded successfully")
    print(f"✓ Adapter loaded: {ADAPTER_PATH}")

## 2. Load Fine-Tuned Model